# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/labs/lab02_heart_disease.ipynb)

# BME i9400 — Lab 2
## A Classifier for Coronary Artery Disease

**Fall 2026 · Released Wednesday, November 11 · Notebook due Wednesday, November 18, 11:00 AM EST (start of class) · Defense quiz Wednesday, November 18, in class**

In this lab you build and evaluate a clinical classifier on your own: prepare a dataset with missing
values and categorical codes, fit logistic regression by gradient descent and diagnose the run,
choose a penalty by cross-validation, estimate the performance of the whole procedure by nested
cross-validation, choose a decision threshold on out-of-fold probabilities, and
evaluate the result once on a test set. The methods are those of Meetings 8 to 13 and 17. The
dataset is new.

**How the lab is assessed.** The lab component of the course is 20% of the grade, over three labs.
Within a lab, the in-class defense quiz counts 70% and the submitted notebook 30%.

- **The notebook** is graded on whether it runs from top to bottom without error, whether every task
  is complete and correct, and on the quality of the report in section 9.
- **The defense quiz** is a ten-minute written quiz at the start of the meeting on Wednesday,
  November 18, taken closed-book: no laptop, no notes, no AI tools. It asks you to explain and
  justify what is in *your* notebook. Section 10 describes it.

**Generative AI is permitted and expected** on the notebook. You are responsible for every line you
submit, and the defense tests whether you can account for it without assistance. Record how you used
AI in section 10.

**Work individually.** Discussion of approach is encouraged; code and text must be your own.

---
## 1 — The question and the data

Coronary artery disease is diagnosed definitively by **coronary angiography**: a catheter is passed
to the heart, contrast dye is injected, and the coronary arteries are imaged by X-ray. Angiography
is invasive, expensive, and carries a small risk of serious complications. Before it is ordered, a
physician has the patient's history, a resting ECG, routine blood tests, and usually an exercise
stress test. The question of this lab is the question of Detrano et al. (*American Journal of
Cardiology*, 1989): **how well can the result of angiography be predicted from the information
gathered before it?**

### The data

303 patients referred for coronary angiography at the Cleveland Clinic Foundation between 1981 and
1984, collected by Robert Detrano and distributed by the UCI Machine Learning Repository as the
"Heart Disease" dataset. The course copy is `data/heart_cleveland.csv`, one row per patient, with
the columns renamed for readability and the values otherwise as distributed.

| Column | Meaning | Values |
|---|---|---|
| `age` | age | years |
| `sex` | sex | 1 = male, 0 = female |
| `chest_pain` | type of chest pain | 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = no chest pain |
| `resting_bp` | resting systolic blood pressure on admission | mm Hg |
| `cholesterol` | serum cholesterol | mg/dl |
| `fasting_sugar` | fasting blood sugar above 120 mg/dl | 1 = yes, 0 = no |
| `rest_ecg` | resting ECG | 0 = normal, 1 = ST-T wave abnormality, 2 = left ventricular hypertrophy |
| `max_hr` | maximum heart rate reached during the exercise test | beats per minute |
| `exercise_angina` | chest pain induced by the exercise test | 1 = yes, 0 = no |
| `st_depression` | depression of the ST segment during exercise, relative to rest | mm |
| `st_slope` | slope of the ST segment at peak exercise | 1 = upsloping, 2 = flat, 3 = downsloping |
| `vessels` | number of major coronary vessels visible on fluoroscopy | 0 to 3 |
| `thal` | thallium scintigraphy (an imaging stress test) | 3 = normal, 6 = fixed defect, 7 = reversible defect |
| `num` | **the source of the target**: angiography result | 0 = no vessel with ≥ 50% narrowing; 1 to 4 = disease |

> The **target** of this lab is binary: a patient has coronary artery disease if `num` is 1 or more,
> that is, if at least one major vessel is narrowed by 50% or more of its diameter. The levels 1 to
> 4 are not used separately; the documentation does not define them consistently, and the published
> analyses of these data use presence against absence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

URL = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/heart_cleveland.csv"
raw = pd.read_csv(URL)

print(f"{len(raw)} patients, {raw.shape[1]} columns")
print(raw.dtypes)
raw.head()

---
## 2 — Data readiness

### 2.1 Missing values

The output above shows that two columns were read as text (`object`) rather than as numbers. In this
file a missing value is recorded as the character `?`, and a single `?` makes pandas read the whole
column as text.

Reload the file so that every `?` becomes `NaN` (the `na_values` argument of `pd.read_csv`), count the
missing values in each column, and construct the binary target `y` from `num`. The features `X` are
every column except `num`.

In [ ]:
# --- TODO: reload with missing values marked, and build X and y ---
df = None                 # DataFrame: all 14 columns, every column numeric, "?" replaced by NaN
X  = None                 # DataFrame: the 13 feature columns
y  = None                 # NumPy array of 0/1: 1 if num >= 1

assert df is not None and X is not None and y is not None, "Fill in df, X and y."
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns), "Every column should be numeric."
assert "num" not in X.columns and X.shape == (303, 13), "X should hold the 13 features and not num."
assert set(np.unique(y)) == {0, 1} and y.sum() == 139, "y should be 1 for the 139 patients with num >= 1."
print(df.isna().sum()[lambda s: s > 0])
print(f"prevalence of disease: {y.mean():.3f}    women: {(X['sex'] == 0).sum()}")

### 2.2 Codes that are not quantities

Some columns hold numbers that are labels for categories. Entering such a column into logistic
regression as a single number assumes that the categories are ordered and equally spaced: for
`thal`, that a reversible defect (7) is "one unit more" than a fixed defect (6), and that both are
far from normal (3) in proportion to the distance between the codes.

> The standard remedy is **one-hot encoding**: a column with $k$ categories is replaced by binary
> columns, one per category, each equal to 1 for the patients in that category. Each category then
> receives its own weight.
>
> In a model with an intercept, one of the $k$ columns is dropped. The $k$ columns always sum to 1,
> which is the intercept column, so keeping all of them makes the weights non-unique. The dropped
> category is the **reference category**, and the weight of every other category measures its
> difference from the reference.

List the columns that should be one-hot encoded. A column that is already binary (0/1) does not need
to be: its single weight already describes the difference between its two categories.

In [ ]:
# --- TODO: the categorical columns with more than two categories ---
categorical = None        # a list of column names

assert categorical is not None, "Fill in categorical."
assert set(categorical) == {"chest_pain", "rest_ecg", "st_slope", "thal"}, \
    "Look again at the table in section 1: which columns are labels with three or more categories?"
binary  = ["sex", "fasting_sugar", "exercise_angina"]
numeric = [c for c in X.columns if c not in categorical + binary]
print("numeric:    ", numeric)
print("binary:     ", binary)
print("categorical:", categorical)

The preprocessing is done for you, as one object that treats each group of columns differently:

- **numeric** columns: missing values replaced by the median, then standardised (Meeting 10)
- **binary** columns: passed through unchanged, so that each weight compares "yes" with "no"
- **categorical** columns: missing values replaced by the most frequent category, then one-hot
  encoded with the first category as the reference (`drop="first"`): typical angina for
  `chest_pain`, normal for `rest_ecg`, upsloping for `st_slope`, normal for `thal`

> `ColumnTransformer` applies a different pipeline to each list of columns and places the results
> side by side. Like every step of a pipeline, it learns its medians, means, standard deviations and
> categories from the data passed to `fit`, and only applies them in `transform`. The categories
> are the one exception here: they are given in advance, from the documentation, because
> `rest_ecg = 1` occurs in only four patients and is absent from some cross-validation folds.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# The categories of each code, from the table in section 1, so that a category absent from one fold still gets a column.
codes = {"chest_pain": [1, 2, 3, 4], "rest_ecg": [0, 1, 2], "st_slope": [1, 2, 3], "thal": [3, 6, 7]}

def preprocessor():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("bin", "passthrough", binary),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(categories=[codes[c] for c in categorical], drop="first")), categorical),
    ])

print(preprocessor().fit(X).get_feature_names_out())

### 2.3 What is known when the decision is made

The model is meant to be used **before** angiography is ordered. For each of `vessels` and `thal`,
state which procedure produces the value, and whether it would be available at that point. Use the
table in section 1 and, if needed, a reference on cardiac fluoroscopy and thallium scintigraphy. Do
not remove the columns: the rest of the lab uses all 13 features, and section 9 asks you to state the
consequence.

> Meeting 1: a feature is usable only if it exists, with the same meaning, at the moment the model is
> applied. A feature produced by the procedure that measures the target is a form of **leakage**.

**Answer 2.3** — For `vessels` and `thal`: the procedure that produces it, and whether it is available
before angiography. *(Replace this text.)*

---
## 3 — Splitting the data

The test split is fixed below, stratified by the target, so that every student's test set is the
same 76 patients.

> **Stratified** splitting keeps the prevalence of disease the same in each part. With 303 patients,
> an unstratified split can by chance put noticeably more or fewer diseased patients in the test set.

Inside the development set, section 4 needs one validation set, and sections 5 to 7 use
cross-validation instead. Make the validation split below: stratified, about 25% of the development
set, with a fixed `random_state` of your choice.

In [ ]:
from sklearn.model_selection import train_test_split

# Done for you: the test set, 25% of the patients, identical for every student.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=9400)

# --- TODO: a stratified split of the development set into training and validation sets (about 25% validation) ---
X_train, X_val, y_train, y_val = None, None, None, None

assert X_train is not None, "Make the validation split."
assert len(X_train) + len(X_val) == len(X_dev) and len(X_val) >= 0.2 * len(X_dev), \
    "Validation should be about a quarter of the development set."
assert abs(y_train.mean() - y_val.mean()) < 0.02, "Stratify the split by y_dev."
print(f"training {len(y_train)}   validation {len(y_val)}   test {len(y_test)}")
print(f"prevalence: training {y_train.mean():.3f}   validation {y_val.mean():.3f}   test {y_test.mean():.3f}")

---
## 4 — Logistic regression by gradient descent

Logistic regression has no closed-form solution; its weights are found by iterative optimisation
(Meeting 13). In this section you write the optimiser yourself, without a penalty, and diagnose its
behaviour.

The design matrices are built for you: the preprocessor is fitted on the training set only, and a
column of ones is prepended for the intercept.

### 4.1 The loss and its gradient

With $\mathbf{w}$ the weight vector (intercept first), $\mathbf{x}_i$ the $i$-th row of the design
matrix, and $\hat p_i = \sigma(\mathbf{w}^\top \mathbf{x}_i)$ the predicted probability, the mean
cross-entropy loss and its gradient are (Meeting 8)

$$L(\mathbf{w}) = -\frac{1}{n}\sum_{i=1}^{n}\Big[\,y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i)\Big]
\qquad\qquad
\nabla L(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n} (\hat p_i - y_i)\,\mathbf{x}_i = \frac{1}{n}\,\mathbf{X}^\top(\hat{\mathbf{p}} - \mathbf{y})$$

> $\sigma(z) = 1/(1+e^{-z})$ is the logistic (sigmoid) function.
>
> Clip $\hat p$ to $[10^{-12},\, 1-10^{-12}]$ before taking the logarithm, so that a probability that
> rounds to exactly 0 or 1 does not produce `inf`.

In [ ]:
prep_gd = preprocessor().fit(X_train)
A_train = np.column_stack([np.ones(len(X_train)), prep_gd.transform(X_train)])     # n_train x (1 + 18)
A_val   = np.column_stack([np.ones(len(X_val)),   prep_gd.transform(X_val)])
print(f"design matrices: training {A_train.shape}, validation {A_val.shape}")

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# --- TODO: the mean cross-entropy loss and its gradient ---
def loss(w, A, y):
    return None

def gradient(w, A, y):
    return None

# Checks: the loss at w = 0 is log 2, and the gradient agrees with a finite-difference estimate.
w0 = np.zeros(A_train.shape[1])
assert loss(w0, A_train, y_train) is not None, "Write loss()."
assert np.isclose(loss(w0, A_train, y_train), np.log(2)), "At w = 0 every probability is 0.5, so the loss is log 2."
rng = np.random.default_rng(0); w1 = 0.3 * rng.standard_normal(A_train.shape[1]); e = np.eye(len(w1))[5] * 1e-6
fd = (loss(w1 + e, A_train, y_train) - loss(w1 - e, A_train, y_train)) / 2e-6
assert np.isclose(gradient(w1, A_train, y_train)[5], fd, rtol=1e-4), "The gradient does not match the loss."
print("loss and gradient pass")

### 4.2 The learning rate

For least squares, gradient descent is stable for learning rates below $2/h_{\max}$, where $h_{\max}$
is the largest eigenvalue of the Hessian (Meeting 13, section 3). The Hessian of the logistic loss is

$$\mathbf{H}(\mathbf{w}) = \frac{1}{n}\,\mathbf{X}^\top \mathbf{S}\,\mathbf{X}, \qquad \mathbf{S} = \mathrm{diag}\big(\hat p_i(1-\hat p_i)\big)$$

and it changes with $\mathbf{w}$. Because $\hat p(1-\hat p) \le 1/4$, its largest eigenvalue is never
more than $\lambda_{\max}/4$, where $\lambda_{\max}$ is the largest eigenvalue of
$\frac{1}{n}\mathbf{X}^\top\mathbf{X}$. A learning rate that is safe everywhere is therefore

$$\eta_{\text{safe}} = \frac{2}{\lambda_{\max}/4} = \frac{8}{\lambda_{\max}}$$

> $\mathrm{diag}(\cdot)$ is the diagonal matrix with the listed values on its diagonal and zeros
> elsewhere.

Compute $\eta_{\text{safe}}$. Then run gradient descent from $\mathbf{w} = \mathbf{0}$ for 3000 steps
at each of the four learning rates $0.1\,\eta_{\text{safe}},\ \eta_{\text{safe}},\ 3\,\eta_{\text{safe}},\
10\,\eta_{\text{safe}}$, recording the training **and** validation loss after every step. Plot both
curves against the step number for each learning rate (a logarithmic step axis is helpful), and
compare the final weights at $\eta_{\text{safe}}$ with unpenalised `LogisticRegression`.

In [ ]:
from sklearn.linear_model import LogisticRegression

# --- TODO: eta_safe, then gradient descent at four learning rates ---
eta_safe = None
etas     = None           # the four learning rates, in increasing order
history  = None           # a dict: history[eta] = (training losses, validation losses, final weights), each loss an array of 3000

assert eta_safe is not None and history is not None, "Fill in eta_safe and history."
assert np.isclose(eta_safe, 8 / np.linalg.eigvalsh(A_train.T @ A_train / len(A_train)).max()), "eta_safe = 8 / lambda_max."
assert len(history) == 4 and all(len(history[e][0]) == 3000 for e in etas), "Four runs of 3000 steps."

sk = LogisticRegression(penalty=None, max_iter=20000).fit(A_train[:, 1:], y_train)
w_sk = np.r_[sk.intercept_, sk.coef_[0]]
print(f"eta_safe = {eta_safe:.3f}")
for e in etas:
    tr, va, w = history[e]
    print(f"eta = {e:7.3f}   final training loss {tr[-1]:.4f}   final validation loss {va[-1]:.4f}   "
          f"lowest validation loss {va.min():.4f} at step {va.argmin() + 1}")
print(f"sklearn, no penalty: training loss {loss(w_sk, A_train, y_train):.4f}   validation loss {loss(w_sk, A_val, y_val):.4f}")
assert np.isclose(history[eta_safe][0][-1], loss(w_sk, A_train, y_train), atol=1e-3), \
    "At eta_safe, gradient descent should reach the same training loss as sklearn."

# --- TODO: plot training and validation loss against step, one panel per learning rate ---

**Answer 4** — (a) Describe the four runs: which converge, which oscillate or fail to settle, and why,
in terms of $\eta_{\text{safe}}$ and the curvature of the loss. (b) At $\eta_{\text{safe}}$, at which
step is the validation loss lowest, and why does it rise afterwards while the training loss keeps
falling? (c) Name two ways to obtain a model close to the one at that step. *(Replace this text.)*

---
## 5 — Choosing the model by cross-validation

From here on, scikit-learn's solver replaces your gradient descent, and cross-validation on the
development set replaces the single validation set (Meeting 11). The folds and the model are set up
for you:

- `make_model(C, penalty)`: the preprocessor followed by logistic regression with penalty strength
  $1/C$ (`saga` solver, as in Meeting 12, so that the intercept is not penalised)
- `inner`: five stratified folds for choosing $C$ and the penalty
- `outer`: a second, independent set of five folds for estimating performance (Meeting 12)

### 5.1 Choosing $C$ and the penalty

Fit a `GridSearchCV` on the development set over the grid `Cs` and the two penalties `"l1"` and
`"l2"`, on the folds `inner`, scored by `"roc_auc"`. Report the best combination and its
cross-validated AUC. Then print the **odds ratio** of every feature in the refitted best model.

> The **odds ratio** of a feature is $e^{w}$: the factor by which the odds of disease are multiplied
> when that feature increases by one unit and every other feature is held fixed (Meeting 8). For a
> standardised feature, one unit is one standard deviation in the training data; for a binary
> feature, it is the change from 0 to 1; for a one-hot feature, it is the change from the reference
> category to that category.

In [ ]:
import warnings
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def make_model(C=1.0, penalty="l2"):
    return make_pipeline(preprocessor(),
                         LogisticRegression(C=C, penalty=penalty, solver="saga", max_iter=20000))

Cs    = np.logspace(-3, 1, 13)
inner = StratifiedKFold(5, shuffle=True, random_state=9400)
outer = StratifiedKFold(5, shuffle=True, random_state=12)

# --- TODO: grid search over Cs and the two penalties, fitted on the development set ---
search = None

assert search is not None and hasattr(search, "best_params_"), "Fit the GridSearchCV on X_dev, y_dev."
assert search.n_splits_ == 5 and search.scoring == "roc_auc", "Use the folds inner and the AUC."
print(f"best: {search.best_params_}    cross-validated AUC {search.best_score_:.3f}")

# --- TODO: the odds ratio of every feature in search.best_estimator_, sorted, with the feature names ---

### 5.2 Nested cross-validation, and a neural network

`search.best_score_` was used to choose $C$, so it is an optimistic estimate of the AUC of the
procedure (Meeting 12, section 3). Estimate the AUC without that optimism by nested cross-validation:
cross-validate the whole `search` object on the folds `outer`.

Do the same for the small neural network defined below (Meeting 17): one hidden layer of 16 units,
with its L2 penalty `alpha` chosen inside each outer fold. Report, for both models, the mean and
the standard deviation of the AUC over the five outer folds.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_search = GridSearchCV(
    make_pipeline(preprocessor(), MLPClassifier(hidden_layer_sizes=(16,), max_iter=3000, random_state=0)),
    {"mlpclassifier__alpha": np.logspace(-3, 1, 5)}, cv=inner, scoring="roc_auc")

# --- TODO: nested cross-validation of `search` and of `mlp_search` on the folds outer, scored by "roc_auc" ---
auc_lr  = None            # array of 5 outer-fold AUCs for logistic regression
auc_mlp = None            # ... for the neural network

assert auc_lr is not None and auc_mlp is not None and len(auc_lr) == len(auc_mlp) == 5, "Five outer folds each."
print(f"logistic regression: nested AUC {auc_lr.mean():.3f} ± {auc_lr.std():.3f}   (inner best_score_ {search.best_score_:.3f})")
print(f"neural network:      nested AUC {auc_mlp.mean():.3f} ± {auc_mlp.std():.3f}")
print("per-fold difference (network − logistic):", np.round(auc_mlp - auc_lr, 3))

**Answer 5** — (a) The chosen penalty and $C$; if the penalty is L1, which features it removed.
(b) Interpret the odds ratios of two features, with their units, one of them a one-hot category
(state its reference category). (c) Compare the nested AUC with `best_score_`. Which of the two
estimates the performance of the procedure on new patients, and why? (d) Is the neural network
better than logistic regression on these data? Base the answer on the per-fold differences and the
fold-to-fold spread, and say which model you would carry forward and why. *(Replace this text.)*

---
## 6 — Out-of-fold probabilities

The threshold of section 7 is chosen on probabilities for patients the model has not seen. Compute
**out-of-fold** probabilities for the development set: `cross_val_predict` of `search` on the folds
`outer`, so that every patient's probability comes from a model that did not see that patient, and
the choice of $C$ is repeated inside each fold.

> The printed **Brier score** is the mean squared difference between the predicted probability and
> the outcome (0 or 1). Lower is better. Predicting the prevalence $\pi$ for everyone gives
> $\pi(1-\pi)$.

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

# --- TODO: out-of-fold probabilities of `search` on the development set, using the folds outer ---
p_oof = None              # array of len(y_dev): the predicted probability of disease for each patient

assert p_oof is not None and p_oof.shape == y_dev.shape and 0 <= p_oof.min() and p_oof.max() <= 1, \
    "One probability per development patient."
prev = y_dev.mean()
print(f"out-of-fold AUC {roc_auc_score(y_dev, p_oof):.3f}")
print(f"Brier score {brier_score_loss(y_dev, p_oof):.3f}    predicting the prevalence: {prev * (1 - prev):.3f}")

---
## 7 — Choosing a threshold

Missing a patient with coronary artery disease is more harmful than sending a patient without it to
angiography. Choose the threshold $t^*$ as the **largest** threshold at which the out-of-fold
sensitivity on the development set is at least **0.90**, and report the out-of-fold specificity
there (Meeting 10, section 5).

> Choosing the largest such threshold gives the highest specificity among the thresholds that meet
> the sensitivity requirement.
>
> If you use `roc_curve`, pass `drop_intermediate=False`. By default it omits thresholds that do not
> change the shape of the curve, and the threshold you need can be one of them.

In [ ]:
from sklearn.metrics import roc_curve

# --- TODO: the largest threshold with out-of-fold sensitivity >= 0.90, and the specificity there ---
t_star   = None
sens_oof = None
spec_oof = None

assert t_star is not None and sens_oof is not None and spec_oof is not None, "Fill in t_star, sens_oof, spec_oof."
assert np.isclose(sens_oof, (p_oof[y_dev == 1] >= t_star).mean()), "sens_oof should be the sensitivity at t_star."
assert sens_oof >= 0.90 and (p_oof[y_dev == 1] >= np.nextafter(t_star, 1)).mean() < 0.90, \
    "t_star should be the largest threshold that keeps sensitivity at or above 0.90."
print(f"t* = {t_star:.3f}    out-of-fold sensitivity {sens_oof:.3f}    specificity {spec_oof:.3f}")

**Answer 7** — Why the threshold is chosen on out-of-fold probabilities rather than on the training
fit or on the test set, and why $t^*$ is well below 0.5. *(Replace this text.)*

---
## 8 — The test set, once

`search` was refitted on the whole development set by `GridSearchCV` (its default, `refit=True`),
with the $C$ and penalty of section 5.1. Apply it to the test set once, at the threshold $t^*$, and
report in one table: the AUC, sensitivity, specificity, PPV and NPV, alongside the out-of-fold values
from sections 5 and 7 where they exist.

The test-set prevalence, 46%, is that of patients already referred for angiography at a specialist
centre. Among patients with chest pain seen in primary care, the prevalence of significant coronary
artery disease is closer to **10%**. Using Bayes' rule (Meeting 2), compute the PPV and NPV that the
test-set sensitivity and specificity would give at a prevalence of 10%.

> Sensitivity and specificity are properties of the classifier at a threshold; PPV and NPV also
> depend on the prevalence (Meeting 2, section 4).

In [ ]:
# --- TODO: test-set probabilities, the confusion counts at t_star, and the five measures ---
p_test = None
test   = None             # a dict with the keys "AUC", "sensitivity", "specificity", "PPV", "NPV"

assert p_test is not None and test is not None and set(test) == {"AUC", "sensitivity", "specificity", "PPV", "NPV"}, \
    "Fill in p_test and the five measures."
assert np.isclose(test["AUC"], roc_auc_score(y_test, p_test)), "AUC on the test set."

# --- TODO: PPV and NPV at a prevalence of 0.10, from the test sensitivity and specificity ---
def ppv_npv(sens, spec, prevalence):
    return None, None

ppv10, npv10 = ppv_npv(test["sensitivity"], test["specificity"], 0.10)
assert ppv10 is not None, "Write ppv_npv()."
assert np.isclose(ppv_npv(test["sensitivity"], test["specificity"], y_test.mean())[0], test["PPV"], atol=1e-9), \
    "At the test-set prevalence, ppv_npv() should reproduce the test PPV."

summary = pd.DataFrame({
    "out-of-fold (development)": [roc_auc_score(y_dev, p_oof), sens_oof, spec_oof, np.nan, np.nan],
    "test, prevalence 46%":      [test[k] for k in ["AUC", "sensitivity", "specificity", "PPV", "NPV"]],
    "test, prevalence 10%":      [np.nan, test["sensitivity"], test["specificity"], ppv10, npv10],
}, index=["AUC", "sensitivity", "specificity", "PPV", "NPV"])
print(summary.round(3).to_string())

The test set is not used again after this cell. If you change a model or the threshold after seeing
these numbers, the test set is no longer a test set.

---
## 9 — Report

Write the report in the cell below, in at most 300 words, under these headings. It is read by someone
who has not seen the rest of the notebook.

1. **The task.** What a row of $X$ is, what $y$ is, and what population the data describe.
2. **Data decisions.** How missing values and categorical codes were handled, and why.
3. **The model.** The model chosen, its penalty, and why it was preferred to the neural network.
4. **Performance.** The nested cross-validated AUC and the test-set results at $t^*$, with one
   sentence on whether they agree.
5. **What the threshold means.** For 100 referred patients like those in the test set, how many with
   disease are missed and how many without disease are sent to angiography; and what changes at a
   prevalence of 10%.
6. **Limitations.** At least three, including the consequence of section 2.3 and the population.

### Report

**1. The task.** *(Replace this text.)*

**2. Data decisions.**

**3. The model.**

**4. Performance.**

**5. What the threshold means.**

**6. Limitations.**

---
## 10 — AI use, submission, and the defense

### AI use statement

In the cell below, list the AI tools you used and what for — code, debugging, explanation, writing —
in two to five lines. The statement is not graded for its content; its absence costs credit.

**AI use statement** — *(Replace this text.)*

### Submitting

**Due Wednesday, November 18, at 11:00 AM EST — the start of class.**

1. `Runtime ▸ Restart session and run all`. Every cell must run without error, and every check must
   pass.
2. `File ▸ Download ▸ Download .ipynb`.
3. Rename the file to `YOUR-GITHUB-USERNAME_lab02.ipynb` and move it into the `checkins/` folder of
   your clone.
4. Commit in GitHub Desktop, push, and open a pull request.

Late notebooks are accepted for 48 hours with a 20% penalty on the notebook portion. The defense quiz
cannot be made up except for a documented excused absence (syllabus).

### The defense quiz — Wednesday, November 18

Ten minutes, written, at the start of the meeting, closed-book: laptops closed, no notes, no AI tools.
You will not have your notebook in front of you, so know its contents before you arrive, including your
main results to about two significant figures. Five or six short questions about your own notebook,
each answerable in two or three sentences. The kinds of question to expect:

- why a step was necessary — why `thal` is one-hot encoded, why the imputer sits inside the pipeline,
  why the threshold is chosen on out-of-fold probabilities
- what a curve shows — what happened at $10\,\eta_{\text{safe}}$, and why the validation loss rose
  while the training loss fell
- what a number means — an odds ratio, with its units and reference category; your nested AUC
- what would have happened under a different choice — at a prevalence of 10%, or if `vessels` were
  removed

Each question is scored 0, 1 or 2. Prepare by being able to explain every cell of your notebook
without looking at it.